# Figure S5 — EHR volcano plots (Mount Sinai + UK Biobank)

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Mount Sinai volcano

In [ ]:

ms = io.read_ehr_ms().dropna(subset=["logit_or", "logit_p"])
ms["neglog10p"] = -np.log10(ms["logit_p"].clip(lower=1e-300))
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.scatter(ms["logit_or"], ms["neglog10p"], s=4, alpha=0.3, c="#4C72B0")
ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel("logit OR")
ax.set_ylabel("-log10 p")
ax.set_title("Fig S5 — Mount Sinai")
fig.tight_layout()
out = style.save_panel(fig, "figS5_ms_volcano", ms[["Drug Name", "ICD10", "logit_or", "logit_p", "neglog10p"]])
plt.show()
print(out)


## UK Biobank volcano

In [ ]:

uk = io.read_ehr_ukb().dropna(subset=["odds_ratio"])
# derive a pseudo p from counts if needed; plot OR vs exposure support
uk["log_or"] = np.log(uk["odds_ratio"].clip(lower=1e-6))
uk["support"] = uk[["drug_cancer", "drug_no_cancer"]].sum(axis=1)
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.scatter(uk["log_or"], np.log10(uk["support"] + 1), s=8, alpha=0.4, c="#DD8452")
ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel("log odds ratio")
ax.set_ylabel("log10(exposed n)")
ax.set_title("Fig S5 — UK Biobank")
fig.tight_layout()
out = style.save_panel(fig, "figS5_ukb_volcano", uk[["Drug Name", "ICD10", "odds_ratio", "log_or", "support"]])
plt.show()
print(out)
